## 1. Packages

In [3]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import mean_absolute_error

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset, random_split

import keras 

import optuna

## 2. Import des données

In [4]:
x = pd.read_csv(
    "data/x_train.csv", low_memory=False
)  # par défaut, low_memory vaut true, cela fait que le fichier est lu par morceau mais ça peut provoquer des erreurs de type de données
y = pd.read_csv("data/y_train.csv", low_memory=False)
x_test_final = pd.read_csv("data/x_test.csv", low_memory=False)

## 3. Préparation des données

### 3.1 Suppression des colones inutiles

In [5]:
x = x.drop(columns=["Unnamed: 0"], errors="ignore")
x = x.drop(columns=["Unnamed: 0.1"], errors="ignore")
y = y.drop(columns=["Unnamed: 0"], errors="ignore")
x_test_final = x_test_final.drop(columns=["Unnamed: 0"], errors="ignore")
x_test_final = x_test_final.drop(columns=["Unnamed: 0.1"], errors="ignore")

x.head()

,train,gare,date,arret,p2q0,p3q0,p4q0,p0q2,p0q3,p0q4
0,VBXNMF,KYF,2023-04-03,8,0.0,0.0,1.0,-3.0,-1.0,-2.0
1,VBXNMF,JLR,2023-04-03,9,0.0,0.0,0.0,1.0,0.0,1.0
2,VBXNMF,EOH,2023-04-03,10,-1.0,0.0,0.0,-1.0,0.0,0.0
3,VBXNMF,VXY,2023-04-03,11,-1.0,-1.0,0.0,2.0,-2.0,0.0
4,VBXNMF,OCB,2023-04-03,12,-1.0,-1.0,-1.0,-1.0,3.0,2.0


### 3.2 Encodage de la gare
Convertion des identifiants des gares (sous forme de chaîne de caractère) en données numériques. Ici il s'agit de label encoding. Le hot encoding a également été testé, mais les résultats n'étaient pas meilleurs.

In [6]:
label_encoder = LabelEncoder()

all_gare = pd.concat([
    x["gare"].astype(str),
    x_test_final["gare"].astype(str)
])

label_encoder.fit(all_gare)

x["gare"] = label_encoder.transform(x["gare"].astype(str))
x_test_final["gare"] = label_encoder.transform(x_test_final["gare"].astype(str))

x.head()

,train,gare,date,arret,p2q0,p3q0,p4q0,p0q2,p0q3,p0q4
0,VBXNMF,34,2023-04-03,8,0.0,0.0,1.0,-3.0,-1.0,-2.0
1,VBXNMF,26,2023-04-03,9,0.0,0.0,0.0,1.0,0.0,1.0
2,VBXNMF,14,2023-04-03,10,-1.0,0.0,0.0,-1.0,0.0,0.0
3,VBXNMF,68,2023-04-03,11,-1.0,-1.0,0.0,2.0,-2.0,0.0
4,VBXNMF,43,2023-04-03,12,-1.0,-1.0,-1.0,-1.0,3.0,2.0


### 3.3 Conversion de date
Conversion de la date en format datetime utilisable pour panda

1. Décomposition de la colone date en 2 colones : day et month. L'année n'est pas conservée car toutes les données d'entrainement et de test ont pour année 2023.

2. La date sous format chaîne de caractère n'est en effet pas pertinente pour saisir des corrélations temporelles (tendances mensuelles, journalière)


In [7]:
try:
    x["month"] = pd.to_datetime(x["date"]).dt.month
    x["day"] = pd.to_datetime(x["date"]).dt.day
    
    x_test_final["month"] = pd.to_datetime(x_test_final["date"]).dt.month
    x_test_final["day"] = pd.to_datetime(x_test_final["date"]).dt.day

except Exception as e:
    print(f"Erreur: {e}")

### 3.4 Encodage cyclique des dates
On transforme les colonnes "month" et "day" en représentations cycliques à l’aide des fonctions sinus et cosinus. Cela permet de conserver l’information cyclique du calendrier (par exemple, janvier et décembre restent proches).

In [8]:
x["month_sin"] = np.sin(2 * np.pi * x["month"] / 12)
x["month_cos"] = np.cos(2 * np.pi * x["month"] / 12)
x = x.drop(columns=["month"])

x["day_sin"] = np.sin(2 * np.pi * x["day"] / 31)
x["day_cos"] = np.cos(2 * np.pi * x["day"] / 31)
x = x.drop(columns=["day"])

x_test_final["month_sin"] = np.sin(2 * np.pi * x_test_final["month"] / 12)
x_test_final["month_cos"] = np.cos(2 * np.pi * x_test_final["month"] / 12)
x_test_final = x_test_final.drop(columns=["month"])

x_test_final["day_sin"] = np.sin(2 * np.pi * x_test_final["day"] / 31)
x_test_final["day_cos"] = np.cos(2 * np.pi * x_test_final["day"] / 31)
x_test_final = x_test_final.drop(columns=["day"])


In [9]:
# Supprimer la colonne date apres la conversion
x = x.drop(columns=["date"])
x_test_final = x_test_final.drop(columns=["date"])

In [10]:
x.head(), y.head()

(    train  gare  arret  p2q0  p3q0  p4q0  p0q2  p0q3  p0q4  month_sin  \
 0  VBXNMF    34      8   0.0   0.0   1.0  -3.0  -1.0  -2.0   0.866025   
 1  VBXNMF    26      9   0.0   0.0   0.0   1.0   0.0   1.0   0.866025   
 2  VBXNMF    14     10  -1.0   0.0   0.0  -1.0   0.0   0.0   0.866025   
 3  VBXNMF    68     11  -1.0  -1.0   0.0   2.0  -2.0   0.0   0.866025   
 4  VBXNMF    43     12  -1.0  -1.0  -1.0  -1.0   3.0   2.0   0.866025   
 
    month_cos   day_sin   day_cos  
 0       -0.5  0.571268  0.820763  
 1       -0.5  0.571268  0.820763  
 2       -0.5  0.571268  0.820763  
 3       -0.5  0.571268  0.820763  
 4       -0.5  0.571268  0.820763  ,
    p0q0
 0  -1.0
 1  -1.0
 2  -1.0
 3   1.0
 4   3.0)

In [11]:
x_test_final.head()

,train,gare,arret,p2q0,p3q0,p4q0,p0q2,p0q3,p0q4,month_sin,month_cos,day_sin,day_cos
0,ZPQEKP,68,12,0.0,0.0,-2.0,-4.0,-2.0,-4.0,-0.5,0.866025,0.485302,-0.874347
1,KIQSRA,68,12,0.0,0.0,-1.0,1.0,-1.0,0.0,-0.5,0.866025,0.485302,-0.874347
2,QQJYYT,68,12,0.0,1.0,-1.0,1.0,-1.0,1.0,-0.5,0.866025,0.485302,-0.874347
3,FVKYMZ,68,12,0.0,0.0,-1.0,-1.0,0.0,-1.0,-0.5,0.866025,0.485302,-0.874347
4,GXNZBY,5,12,1.0,-2.0,0.0,0.0,0.0,0.0,-0.5,0.866025,0.485302,-0.874347


### 3.5 Suppression de la colone train
On enlève la colone train car il s'agit d'un identifiant unique par jour (ça n'a pas de sens de faire des catégories dans ce cas car le nombre de jours est très grand).

In [12]:
x = x.drop(columns=["train"], errors="ignore")
x_test_final = x_test_final.drop(columns=["train"])
x.head()

,gare,arret,p2q0,p3q0,p4q0,p0q2,p0q3,p0q4,month_sin,month_cos,day_sin,day_cos
0,34,8,0.0,0.0,1.0,-3.0,-1.0,-2.0,0.866025,-0.5,0.571268,0.820763
1,26,9,0.0,0.0,0.0,1.0,0.0,1.0,0.866025,-0.5,0.571268,0.820763
2,14,10,-1.0,0.0,0.0,-1.0,0.0,0.0,0.866025,-0.5,0.571268,0.820763
3,68,11,-1.0,-1.0,0.0,2.0,-2.0,0.0,0.866025,-0.5,0.571268,0.820763
4,43,12,-1.0,-1.0,-1.0,-1.0,3.0,2.0,0.866025,-0.5,0.571268,0.820763


## 4. Entrainement du modèle

### 4.1 Préparation de dataset
#### 4.1.1 Séparation des variables catégorielles et numériques
On sépare les données en deux parties : les identifiants des gares (station_ids) et les autres caractéristiques numériques (numeric_features). Cela permet de traiter différemment les variables catégorielles et les variables continues.

In [13]:
station_ids = x["gare"].values          # shape: (N,)
numeric_features = x.drop(columns=["gare"]).values

station_ids_test_final = x_test_final["gare"].values          # shape: (N,)
numeric_features_test_final = x_test_final.drop(columns=["gare"]).values

In [14]:
print(numeric_features[1]), print(numeric_features_test_final.shape)

[ 9.          0.          0.          0.          1.          0.
  1.          0.8660254  -0.5         0.57126822  0.82076344]
(20657, 11)


(None, None)

#### 4.1.2 Normalisation des données
On normalise uniquement les données numériques

In [15]:
scaler = StandardScaler()
numeric_features = scaler.fit_transform(numeric_features)

numeric_features_test_final = scaler.transform(numeric_features_test_final)

In [16]:
print(numeric_features_test_final.shape)

(20657, 11)


#### 4.1.3 Conversion des données en tenseurs PyTorch
On convertit les données préparées en tenseurs PyTorch afin de pouvoir les utiliser directement dans un modèle. Les identifiants des gares (station_ids) sont convertis en tenseurs d’entiers longs pour les embeddings, tandis que les autres caractéristiques numériques et les labels (y) sont convertis en tenseurs flottants pour l’entraînement.

In [17]:
station_ids = torch.tensor(station_ids, dtype=torch.long)
numeric_features = torch.tensor(numeric_features, dtype=torch.float32)
y = torch.tensor(y.values, dtype=torch.float32)

station_ids_test_final = torch.tensor(station_ids_test_final, dtype=torch.long)
numeric_features_test_final = torch.tensor(numeric_features_test_final, dtype=torch.float32)

#### 4.1.4 Création d’un TensorDataset pour l’entraînement

In [18]:
dataset = TensorDataset(station_ids, numeric_features, y)

#### 4.1.5 Diviser testset et trainset

In [20]:
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size

train_dataset, test_dataset = random_split(dataset, [train_size, test_size])

train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False)

### 4.2 Le modèle
#### 4.2.1 Détecter l'appareil

In [21]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cpu


#### 4.2.2 Définir le réseau

In [22]:
class MLPWithEmbedding(nn.Module):
    def __init__(self, num_stations, emb_dim, num_numeric_features):
        super().__init__()
        
        # Créer une couche d’embedding qui va transformer chaque identifiant de gare en un vecteur de dimension emb_dim
        self.embedding = nn.Embedding(num_stations, emb_dim)

        # Créer l'architecture de notre reseau neurone
        self.mlp = nn.Sequential(
            nn.Linear(emb_dim + num_numeric_features, 128),
            nn.ReLU(),
            nn.Dropout(0.2), 
            

            nn.Linear(128, 64),
            nn.ReLU(),

            nn.Linear(64, 32),
            nn.ReLU(),
            
            nn.Linear(32, 1)
        )

    def forward(self, station_id, numeric_features):
        # Transformer les identifiants en  un vecteur
        station_emb = self.embedding(station_id)   

        # Concatener chaque identifiant + les autre valeurs numériques
        x = torch.cat([station_emb, numeric_features], dim=1)
        return self.mlp(x)

#### 4.2.3 Intancier le modèle

In [44]:
model = MLPWithEmbedding(
    num_stations = x["gare"].nunique(),
    emb_dim = 8,
    num_numeric_features = numeric_features.shape[1]
)
model.to(device)


criterion = nn.L1Loss()  # MAE
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)


epochs = 30

### 4.3 Train loop

In [45]:
for epoch in range(epochs):
    model.train()
    total_loss = 0
    
    for batch_station, batch_numeric, batch_y in train_loader:

        batch_station = batch_station.to(device)
        batch_numeric = batch_numeric.to(device)
        batch_y = batch_y.to(device)
        
        optimizer.zero_grad()
        pred = model(batch_station, batch_numeric).squeeze()  # shape: (batch,)
        batch_y = batch_y.squeeze()  # shape: (batch,)
        
        loss = criterion(pred, batch_y)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item() * batch_station.size(0)
    
    avg_mae = total_loss / len(dataset)
    train_mae = total_loss / len(train_loader.dataset)

    # Calculer le score sur le testset
    model.eval()
    total_val_loss = 0
    
    with torch.no_grad():
        for batch_station, batch_numeric, batch_y in test_loader:
            batch_station = batch_station.to(device)
            batch_numeric = batch_numeric.to(device)
            batch_y = batch_y.to(device)
            
            pred = model(batch_station, batch_numeric).squeeze()
            batch_y = batch_y.squeeze()
            
            loss = criterion(pred, batch_y)
            total_val_loss += loss.item() * batch_station.size(0)
    
    val_mae = total_val_loss / len(test_loader.dataset)
    
    print(f"Epoch {epoch+1:02d} - Train MAE: {train_mae:.4f} | Val MAE: {val_mae:.4f}")

Epoch 01 - Train MAE: 0.7537 | Val MAE: 0.7134
Epoch 02 - Train MAE: 0.7131 | Val MAE: 0.6985
Epoch 03 - Train MAE: 0.6999 | Val MAE: 0.6871
Epoch 04 - Train MAE: 0.6911 | Val MAE: 0.6808
Epoch 05 - Train MAE: 0.6850 | Val MAE: 0.6762
Epoch 06 - Train MAE: 0.6791 | Val MAE: 0.6737
Epoch 07 - Train MAE: 0.6759 | Val MAE: 0.6680
Epoch 08 - Train MAE: 0.6732 | Val MAE: 0.6691
Epoch 09 - Train MAE: 0.6716 | Val MAE: 0.6639
Epoch 10 - Train MAE: 0.6689 | Val MAE: 0.6619
Epoch 11 - Train MAE: 0.6673 | Val MAE: 0.6628
Epoch 12 - Train MAE: 0.6659 | Val MAE: 0.6596
Epoch 13 - Train MAE: 0.6648 | Val MAE: 0.6592
Epoch 14 - Train MAE: 0.6634 | Val MAE: 0.6568
Epoch 15 - Train MAE: 0.6622 | Val MAE: 0.6566
Epoch 16 - Train MAE: 0.6617 | Val MAE: 0.6566
Epoch 17 - Train MAE: 0.6606 | Val MAE: 0.6568
Epoch 18 - Train MAE: 0.6599 | Val MAE: 0.6546
Epoch 19 - Train MAE: 0.6590 | Val MAE: 0.6538
Epoch 20 - Train MAE: 0.6582 | Val MAE: 0.6536
Epoch 21 - Train MAE: 0.6573 | Val MAE: 0.6534
Epoch 22 - Tr

## 5. Prédiction et création du fichier CSV

In [59]:
final_dataset = TensorDataset(station_ids_test_final, numeric_features_test_final)

In [60]:
final_test_loader = DataLoader(final_dataset, batch_size=256, shuffle=False)

In [83]:
model.to(device)
model.eval()  

preds = []

with torch.no_grad():
    for batch_station, batch_numeric in final_test_loader:
        batch_station = batch_station.to(device)
        batch_numeric = batch_numeric.to(device)
        
        batch_pred = model(batch_station, batch_numeric).squeeze()
        preds.append(batch_pred.cpu())

preds = torch.cat(preds).numpy()


submission = pd.DataFrame({
    "p0q0": preds  
})
submission.to_csv("y_pred_test_nn.csv", index=True, sep=",")
print("y_test_final.csv")

y_test_final.csv


## 6. Finetuning

In [48]:
def objective(trial):
    emb_dim = trial.suggest_int("emb_dim", 4, 8)
    hidden1 = trial.suggest_int("hidden1", 128, 256, step=32)
    hidden2 = trial.suggest_int("hidden2", 64, 128, step=16)
    hidden3 = trial.suggest_int("hidden3", 32, 64, step=8)
    lr = trial.suggest_float("learning_rate", 1e-3, 1e-2, log=True)
    dropout = trial.suggest_float("dropout", 0.0, 0.3)

    class MLPOpt(nn.Module):
        def __init__(self, num_stations, num_numeric):
            super().__init__()
            self.embedding = nn.Embedding(num_stations, emb_dim)
            self.mlp = nn.Sequential(
                nn.Linear(emb_dim + num_numeric, hidden1),
                nn.ReLU(),
                nn.Dropout(dropout),
                
                nn.Linear(hidden1, hidden2),
                nn.ReLU(),
                                
                nn.Linear(hidden2, hidden3),
                nn.ReLU(),
                
                nn.Linear(hidden3, 1)
            )
        def forward(self, station_id, numeric):
            x = torch.cat([self.embedding(station_id), numeric], dim=1)
            return self.mlp(x).squeeze(-1)

    model = MLPWithEmbedding(
        num_stations = x["gare"].nunique(),
        emb_dim = emb_dim,
        num_numeric_features = numeric_features.shape[1]
        )
    model.to(device)


    criterion = nn.L1Loss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    # Moins d'epoch pour un finetunning plus rapide
    epochs = 30
    model.train()
    for epoch in range(epochs):
        total_loss = 0
        for batch_station, batch_numeric, batch_y in train_loader:
            batch_station = batch_station.to(device)
            batch_numeric = batch_numeric.to(device)
            batch_y = batch_y.to(device)
            optimizer.zero_grad()
            pred = model(batch_station, batch_numeric).squeeze()  # shape: (batch,)
            loss = criterion(pred, batch_y.squeeze())
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * batch_station.size(0)

    model.eval()
    
    total_loss = 0
    total_samples = 0
    
    with torch.no_grad():
        for batch_station, batch_numeric, batch_y in test_loader:
            batch_station = batch_station.to(device)
            batch_numeric = batch_numeric.to(device)
            batch_y = batch_y.to(device)
            
            pred = model(batch_station, batch_numeric).squeeze()
            batch_y = batch_y.squeeze()
            
            mae = nn.L1Loss()(pred, batch_y.squeeze()).item()
            loss = nn.L1Loss(reduction='sum')(pred, batch_y)  
            total_loss += loss.item()
            total_samples += batch_station.size(0)

    mae = total_loss / total_samples  
    return mae

In [ ]:
study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=2)  # nombre d’essais rapide pour test

print("Meilleurs hyperparamètres :")
print(study.best_trial.params)

## 7. Combinaison de RNN et réseau de neurone
Principe : On a deux chronologies différentes : les trains passés avant dans une même gare et le retard du train dans les gares précédentes On combine ainsi les résultats de deux LSTM pour prédire le retard :

- input_gare : la variable qui change est la gare. On étudie donc la chronologie d'un même train dans les différentes gares.
- input_train : la variable qui change est le train. On étudie donc l'évolution des retards des trains dans une même gare.

### 7.1 Normalisation des données

In [23]:
scaler = StandardScaler()
x_feature_names = x.columns
x_scaled = scaler.fit_transform(x)
x_test_final_scaled = scaler.transform(x_test_final)
x = pd.DataFrame(x_scaled,columns=x_feature_names)
x_test_final = pd.DataFrame(x_test_final_scaled,columns=x_feature_names)

### 7.2 Gestion des arrets
Cette partie de code est utile pour le RNN. Elle permet pour le rrn étudiant l'évolution du retard du train tout au long de son parcours d'associer au retard de chaque arret, le numéro d'arret associé.

In [24]:
x["arret4"] = x["arret"]-4
x["arret3"] = x["arret"]-3
x["arret2"] = x["arret"]-2
x_test_final["arret4"] = x_test_final["arret"]-4
x_test_final["arret3"] = x_test_final["arret"]-3
x_test_final["arret2"] = x_test_final["arret"]-2

### 7.3 Entrainement

In [28]:
# Recreate the pandas DataFrame from scaled data for indexing
x_df = pd.DataFrame(x_scaled, columns=x_feature_names)

X_rnn_gare = np.stack([
    x[["p0q4", "arret4", "month_sin","month_cos",'day_sin','day_cos']].values,
   x[["p0q3", "arret3","month_sin","month_cos",'day_sin','day_cos']].values,
    x[["p0q2", "arret2","month_sin","month_cos",'day_sin','day_cos']].values,
], axis=1)

X_rnn_train = np.stack([
    x[["p4q0","gare", "month_sin","month_cos",'day_sin','day_cos']].values,
   x[["p3q0", "gare","month_sin","month_cos",'day_sin','day_cos']].values,
    x[["p2q0","gare", "month_sin","month_cos",'day_sin','day_cos']].values,
], axis=1)

def create_model(units_gare,dropout,units_train,units_dense,lr):

    input_gare = keras.Input(shape=(3, 6))
    input_train = keras.Input(shape=(3, 6))

    # LSTM 
    x_gare = keras.layers.LSTM(
        units_gare,
        activation="relu",
        dropout=dropout,
    )(input_gare)

    x_train = keras.layers.LSTM(
        units_train,
        activation="relu",
        dropout=dropout,
    )(input_train)
    
    # Fusion et Dense
    x = keras.layers.Concatenate()([x_gare, x_train])
    x = keras.layers.Dense(units_dense, activation="relu")(x)
    output = keras.layers.Dense(1)(x)
    
    model = keras.Model(
        inputs=[input_gare, input_train],
        outputs=output,
    )
    
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss='mean_absolute_error',
        metrics=["mae"],
    )
    
    return model
model=create_model(20,0.1,20,32,0.01)

In [ ]:
# Entraînement
history = model.fit(
    [X_rnn_gare, X_rnn_train], 
    y, 
    batch_size=256, 
    epochs=100, 
    validation_split=0.2, 
)
plt.plot(history.history['loss'], label='train_loss')
plt.plot(history.history['val_loss'], label='val_loss')
plt.xlabel('Epoch')
plt.ylabel('MAE')
plt.title('Évolution de la MAE à chaque epoch')
plt.legend()
plt.show()

### 7.4 Finetunning

In [30]:
def objective(trial):
    units_gare = trial.suggest_int("units_gare", 8, 64)
    units_train = trial.suggest_int("units_train", 8, 64)
    units_dense = trial.suggest_int("units_dense", 8, 64)
    dropout = trial.suggest_float("dropout", 0.0, 0.3)  
    lr = trial.suggest_float("lr", 1e-3, 1e-2, log=True)    
    model = create_model(units_gare,dropout,units_train,units_dense,lr)

    es = keras.callbacks.EarlyStopping(
        patience=20,
        restore_best_weights=True,
        min_delta=0.001,
    )

    history = model.fit(
        [X_rnn_gare, X_rnn_train],
        y,
        validation_split=0.2,
        epochs=10,
        batch_size=trial.suggest_categorical("batch_size", [32, 64, 128]),
        callbacks=[es],
        verbose=0,
    )

    return min(history.history["val_loss"])

In [ ]:
# Création et optimisation de l'étude
study = optuna.create_study(
    direction="minimize", storage="sqlite:///Optuna/optuna_rnn.db", load_if_exists=True
)
study.optimize(
    objective, n_trials=50
)  # tire aléatoirement des valeurs dans l'intervalle donné pour chaque hyperparamètre
study._storage.flush()

print("Best params:", study.best_trial.params)

### 7.5 Prédiction

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42
)
y_train, y_test = y_train.squeeze(), y_test.squeeze()

In [29]:
X_rnn_gare_test = np.stack([
    x_test_final[["p0q4", "arret4","month_sin","month_cos",'day_sin','day_cos']].values,
   x_test_final[["p0q3", "arret3", "month_sin","month_cos",'day_sin','day_cos']].values,
    x_test_final[["p0q2", "arret2", "month_sin","month_cos",'day_sin','day_cos']].values,
], axis=1)

X_rnn_train_test = np.stack([
    x_test_final[["p4q0","gare", "month_sin","month_cos",'day_sin','day_cos']].values,
   x_test_final[["p3q0", "gare","month_sin","month_cos",'day_sin','day_cos']].values,
    x_test_final[["p2q0","gare", "month_sin","month_cos",'day_sin','day_cos']].values,
], axis=1)
preds = model.predict([X_rnn_gare_test, X_rnn_train_test], batch_size=256) 

# Créer le fichier CSV
submission = pd.DataFrame({
    "p0q0": preds.squeeze() 
})
submission.to_csv("y_pred_test_rnn.csv", index=False, sep=",")
print("Fichier y_pred_test_rnn.csv enregistré !")

81/81 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step
Fichier y_pred_test_rnn.csv enregistré !
